# Toxoplasma background frequency: how often a protein turns up in a pull-down regardless of the bait


**What this is.** When you pull down a tagged protein from *Toxoplasma* and send the beads for mass spectrometry, some proteins come back almost every time: ribosomes, chaperones, tubulin, actin, the abundant glycolytic enzymes. They stick to beads, to antibodies, and to anything hydrophobic. A human version of this list exists (the background frequency, Mellacheruvu et al. 2013) and is used routinely to filter hit lists. No *Toxoplasma* version exists.

This notebook builds one from our own uniform re-search: every *Toxoplasma* PRIDE project searched with the same Sage settings against the same combined FASTA (instruction 69), so the numbers are comparable between projects in a way that author-deposited tables are not.

**What you get.** For every parasite protein, the fraction of independent affinity-purification and proximity-labelling experiments in which it is detected, its typical abundance when detected, and a score that says how surprising it is to see that protein, at that abundance, in a new pull-down.

**Read the validation section before using it.** The whole point of a background table is that real complexes survive it. If they do not, the table is not usable yet, and this notebook says so rather than shipping a filter that throws away the biology.

Built 2026-11-08. Inputs are read-only; nothing on the NAS is written or changed.

In [1]:
import json, os, re, glob
import numpy as np, pandas as pd
from scipy import stats
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
MS_ROOT = '/nas_mnt/carruthers/Einar/parascape_omics/prepared_ms_uniform'
SELECTION = '/nas_mnt/carruthers/Einar/parascape_omics/processing_audit/pride_ms_uniform_selection.json'
OUT = '/media/carruthers/mnt3/claude/repo/starplast/results/pulldown_background_2026_11_08'
REPO = '/media/carruthers/mnt3/claude/repo/starplast'
os.makedirs(OUT, exist_ok=True)
print('uniform search output:', MS_ROOT)
print('exists:', os.path.isdir(MS_ROOT))

uniform search output: /nas_mnt/carruthers/Einar/parascape_omics/prepared_ms_uniform
exists: True


## 1 · What is on disk, and what is excluded

The PRIDE selection lists 203 projects (the rule is in the file: keep anything whose taxon or title names *Toxoplasma* or a *Toxoplasma* protein family). The uniform pipeline is still working through them. A project is finished when it has a `done.json` receipt naming its Sage searches and their output checksums.

Three kinds of project are dropped before anything is counted:

* `unsupported.json` — the pipeline could not search it at all (SCIEX `.wiff`, which needs a converter that does not run on this machine; deposits with no spectra).

* `NOT_COMPARABLE.json` — **hydrogen–deuterium exchange MS**. HDX peptides carry an unknown number of deuteriums, so their masses are shifted by amounts the search does not model; the spectra are kept but the tables are not abundance. Instruction 69 measured it: 1,272 and 1,883 PSMs per run against 10,717–45,527 for ordinary bottom-up runs in the same pipeline. These are `PXD031816` and `PXD023791`; two more HDX projects (`PXD034607`, `PXD039335`) were stopped before they were searched and carry `unsupported.json`.

* a project with a receipt but no readable Sage PSM table.

In [2]:
sel = json.load(open(SELECTION))
selected = {e['project']: e for e in sel['keep']}
print('selection rule:', sel['rule'])
print('selected projects:', len(selected))

rows = []
for proj in sorted(os.listdir(MS_ROOT)):
    d = os.path.join(MS_ROOT, proj)
    if not os.path.isdir(d):
        continue
    receipt = os.path.join(d, 'done.json')
    psm = sorted(glob.glob(os.path.join(d, 'sage_*', 'results.sage.tsv')))
    lfq = sorted(glob.glob(os.path.join(d, 'sage_*', 'lfq.tsv')))
    rows.append(dict(
        project=proj,
        in_selection=proj in selected,
        title=selected.get(proj, {}).get('title', ''),
        taxa=selected.get(proj, {}).get('taxa', ''),
        finished=os.path.exists(receipt),
        unsupported=os.path.exists(os.path.join(d, 'unsupported.json')),
        not_comparable=os.path.exists(os.path.join(d, 'NOT_COMPARABLE.json')),
        n_psm_tables=len(psm), n_lfq_tables=len(lfq),
        psm_path=psm[0] if psm else '', lfq_path=lfq[0] if lfq else '',
    ))
disk = pd.DataFrame(rows)
print('project directories on disk:', len(disk))
print('with a done.json receipt:', int(disk.finished.sum()))
print('unsupported.json:', sorted(disk.loc[disk.unsupported, 'project']))
print('NOT_COMPARABLE.json (HDX):', sorted(disk.loc[disk.not_comparable, 'project']))
disk.groupby(['finished', 'unsupported', 'not_comparable']).size()

selection rule: keep when taxa or title matches toxoplasm|gondii|\bmaf1\b|\b(gra|rop|mic)\d+\b|\bpvm\b|parasitophorous (Einar 2026-11-03: gather all Toxoplasma data); other-organism-only studies need their own proteomes and are left out
selected projects: 203
project directories on disk: 111
with a done.json receipt: 97
unsupported.json: ['PXD003399', 'PXD009604', 'PXD011816', 'PXD011828', 'PXD011829', 'PXD011830', 'PXD011831', 'PXD034607', 'PXD039335', 'PXD052664', 'PXD054994', 'PXD063409']
NOT_COMPARABLE.json (HDX): ['PXD023791', 'PXD031816']


finished  unsupported  not_comparable
False     False        False              2
          True         False             12
True      False        False             95
                       True               2

In [3]:
# run-level detail from each receipt: file names and the acquisition flag the pipeline recorded
det = {}
for proj in disk.loc[disk.finished, 'project']:
    rec = json.load(open(os.path.join(MS_ROOT, proj, 'done.json')))
    runs = rec.get('runs') or []
    # the receipt schema changed mid-queue: older entries key the source file as 'raw'
    names = [r.get('source') or r.get('raw') or r.get('mzml') or '' for r in runs]
    flags = sorted({(r.get('acquisition') or {}).get('flag') for r in runs} - {None})
    det[proj] = dict(n_runs=len(runs), run_names=[n for n in names if n],
                     flags='; '.join(flags),
                     sage_version=(rec.get('tools') or {}).get('sage', ''),
                     fasta=os.path.basename(rec.get('fasta', '')))
disk['n_runs'] = disk.project.map(lambda p: det.get(p, {}).get('n_runs', 0))
disk['flags'] = disk.project.map(lambda p: det.get(p, {}).get('flags', ''))
print('runs behind the finished projects:', int(disk.n_runs.sum()))
print('one FASTA for all of them:',
      sorted({v['fasta'] for v in det.values()}), sorted({v['sage_version'] for v in det.values()}))
disk.loc[disk.finished, ['project', 'n_runs', 'flags']].groupby('flags').size()

runs behind the finished projects: 963
one FASTA for all of them: ['parascape_ms_ME49_human_mouse_contaminants.fasta'] ['sage 0.14.6']


flags
                                              1
DDA                                          89
DDA; possible DIA (no data-dependent MS2)     2
possible TMT (MS3 scans)                      3
timsTOF DDA-PASEF or DIA (not read here)      2

In [4]:
usable = disk[disk.finished & ~disk.unsupported & ~disk.not_comparable
              & (disk.n_psm_tables > 0)].copy().reset_index(drop=True)
excluded = disk[disk.finished & ~disk.project.isin(usable.project)].copy()
print('finished and usable:', len(usable))
print('finished but excluded:', len(excluded))
print('still waiting in the queue:', len(selected) - int(disk.finished.sum()))
excluded[['project', 'title', 'unsupported', 'not_comparable', 'n_psm_tables']]

finished and usable: 94
finished but excluded: 3
still waiting in the queue: 106


,project,title,unsupported,not_comparable,n_psm_tables
12,PXD011440,tgPOFUT2 mediated glycosylation_pft line controls,False,False,0
33,PXD023791,Identification of RON13 binding with with RON4 substrate,False,True,1
42,PXD031816,Conoid extrusion serves as gatekeeper for entry of glideosome components into the pellicular space to control motility and invasion in Apicomplexa,False,True,1


## 2 · Classifying every finished project by what kind of experiment it is

A background table is only meaningful over one kind of experiment. A whole-proteome run detects nearly everything, so mixing it in would make every protein look like a frequent flyer; a phospho-enrichment detects a biased subset. Only affinity purification and proximity labelling go into the background.

PRIDE gives us a title and the deposited file names. The pipeline also recorded an acquisition flag per run (DDA, possible TMT, timsTOF). The rules below are applied in order, first match wins, to the title and to the run file names separately, so it is visible which of the two decided the call. The cue that matched is printed in the output table — that is the audit trail.

**The rules are keyword rules on text a human wrote, not a measurement.** They are printed verbatim so they can be checked and corrected. Where a project matches cues for two classes, or matches only on file names, it is flagged rather than trusted.

In [5]:
# Ordered rules. (class, regex, why). First match wins.
RULES = [
    ('hdx', r'\bhdx\b|hdxms|hydrogen[- ]deuterium',
     'HDX-MS: masses are deuterium-shifted, not abundance (instruction 69)'),
    ('proximity_labelling',
     r'bioid|bira|turboid|turbo-id|\bapex2?\b|proxisome|proximity[- ]?(dependent|label)',
     'BioID / TurboID / APEX: biotin ligase fused to the bait labels neighbours'),
    ('affinity_purification',
     r'immunoprecipit|immuno-precipit|\bco-?ip\b|\bip-?ms\b|interactome|interacting prote|pull[- ]?down|precipitation of|affinity purif|anti-ha ms|associated complex|\binteractors\b|binding with|associates with|interact with|co-?purif|proximal interacting',
     'IP-MS / co-IP / tagged-bait affinity purification'),
    ('ptm_enrichment',
     r'phospho|acetylat|methylscan|methylarginine|lactylation|malonylation|nitrosylation|ubiquitylat|ubiquitinat|succinylat|crotonylat|glycosylat|\bptm\b|post-?translational modification',
     'the sample is an enriched modified subset, not the proteome'),
    ('fractionation',
     r'hyperlopit|\blopit\b|subcellular|fractionat|spatial proteom|organellar|sucrose gradient|\bbn-?page\b',
     'organelle or gradient fractionation: detection depends on the fraction'),
    ('whole_proteome',
     r'proteome|proteomic|protein abundance|expression|abundances following|antigen|immunogenic|proteogenomic',
     'whole-cell or expression proteome'),
]
# File-name cues that are weaker than a title cue but still informative.
NAME_CUES = [
    ('proximity_labelling', r'bioid|bira|turboid|_bio[_.-]|biotin'),
    ('affinity_purification', r'(^|[_.-])ip[_.-]|_ip\d|ha[_.-]?ip|flag[_.-]?ip|beads|pulldown|pull[_-]down|compip'),
    ('hdx', r'hdx'),
    ('ptm_enrichment', r'phos|_pho\d|psn|_sn[_.]|ptm'),
]

def _first_match(text, rules):
    for cls, pat, _why in rules:
        m = re.search(pat, text, re.I)
        if m:
            return cls, m.group(0)
    return '', ''

def _all_matches(text, rules):
    out = []
    for cls, pat, _why in rules:
        m = re.search(pat, text, re.I)
        if m:
            out.append((cls, m.group(0)))
    return out

print('\n'.join(f'{i+1}. {c:22s} {why}' for i, (c, _p, why) in enumerate(RULES)))

1. hdx                    HDX-MS: masses are deuterium-shifted, not abundance (instruction 69)
2. proximity_labelling    BioID / TurboID / APEX: biotin ligase fused to the bait labels neighbours
3. affinity_purification  IP-MS / co-IP / tagged-bait affinity purification
4. ptm_enrichment         the sample is an enriched modified subset, not the proteome
5. fractionation          organelle or gradient fractionation: detection depends on the fraction
6. whole_proteome         whole-cell or expression proteome


In [6]:
cls_rows = []
for _, r in usable.iterrows():
    names = ' '.join(det[r.project]['run_names'])
    t_cls, t_cue = _first_match(r.title, RULES)
    n_cls, n_cue = '', ''
    for cls, pat in NAME_CUES:
        m = re.search(pat, names, re.I)
        if m:
            n_cls, n_cue = cls, m.group(0)
            break
    title_hits = _all_matches(r.title, RULES)
    # the call
    if t_cls:
        call, cue, decided_by = t_cls, f'title: {t_cue!r}', 'title'
    elif n_cls:
        call, cue, decided_by = n_cls, f'file names: {n_cue!r}', 'file names'
    else:
        call, cue, decided_by = 'other', 'no cue matched', 'nothing'
    # confidence
    reasons = []
    if decided_by == 'nothing':
        conf = 'none'
        reasons.append('no keyword matched the title or the file names')
    elif decided_by == 'file names':
        conf = 'low'
        reasons.append('title gave no cue; the call rests on deposited file names')
    else:
        conf = 'high'
    if len(title_hits) > 1:
        other = [c for c, _ in title_hits if c != call]
        if other:
            conf = 'medium' if conf == 'high' else conf
            reasons.append('title also matches ' + '/'.join(sorted(set(other))))
    if n_cls and t_cls and n_cls != t_cls:
        conf = 'medium' if conf == 'high' else conf
        reasons.append(f'file names suggest {n_cls} instead')
    cls_rows.append(dict(project=r.project, title=r.title, n_runs=r.n_runs,
                         acquisition=r.flags, predicted_class=call, cue=cue,
                         decided_by=decided_by, confidence=conf,
                         note='; '.join(reasons), example_runs='; '.join(
                             det[r.project]['run_names'][:4])))
cl = pd.DataFrame(cls_rows)
cl.groupby(['predicted_class', 'confidence']).size().unstack(fill_value=0)

confidence,high,low,medium,none
predicted_class,,,,
affinity_purification,27,5,3,0
other,0,0,0,27
proximity_labelling,6,0,4,0
ptm_enrichment,3,0,8,0
whole_proteome,10,0,1,0


### Hand corrections, each with its reason

Three kinds of project the keyword rules get wrong, checked one at a time against the title and the deposited file names. Every change is listed here with what the rule said and why it is overridden; nothing is corrected silently.

In [7]:
# project -> (class, why). Applied after the rules; recorded in the output table.
OVERRIDES = {
    # Same study, two deposits: PXD039432 is the anti-HA IP, PXD039434 is the biotin
    # (BIO) against vehicle (VEH) TurboID arm. The shared title says neither.
    'PXD039434': ('proximity_labelling',
                  'run names are BIO vs VEH (biotin against vehicle), i.e. proximity labelling, not the IP arm of the same paper'),
    # Titles that name a complex but whose deposit is a gradient/BN-PAGE slice series.
    'PXD068413': ('fractionation',
                  'runs are Top/Middle/Bottom gradient slices of a conoid preparation, not a bait pull-down'),
    'PXD068228': ('fractionation',
                  'runs are Top/Middle/Bottom gradient slices, same preparation style'),
    # Glycan beads capture by affinity to a sugar, not to a tagged parasite protein:
    # there is no bait gene, so it cannot contribute a bait-specific background.
    'PXD079900': ('affinity_purification',
                  'glycan-bead capture: a real affinity purification but the bait is a glycan, so it has no bait gene'),
    # Titles with no keyword at all, resolved from the deposited file names.
    'PXD035658': ('whole_proteome',
                  'run names carry SILAC labelling at two drug concentrations (100uM_L / 10uM_H), i.e. a whole-proteome drug comparison, not a pull-down'),
    'PXD064338': ('whole_proteome',
                  '32 TMT runs of infected host cells (a host immunity study): a whole-proteome experiment'),
}
cl['rule_class'] = cl.predicted_class
for proj, (cls, why) in OVERRIDES.items():
    if proj in set(cl.project):
        i = cl.index[cl.project == proj][0]
        cl.loc[i, 'predicted_class'] = cls
        cl.loc[i, 'confidence'] = 'hand-checked'
        cl.loc[i, 'note'] = (cl.loc[i, 'note'] + '; ' if cl.loc[i, 'note'] else '') + \
            f'override of rule call {cl.loc[i, "rule_class"]!r}: {why}'
cl['class'] = cl.predicted_class
print(cl.groupby('class').size().to_string())
print()
print('needs a human eye (confidence low/none/medium):',
      int(cl.confidence.isin(['low', 'none', 'medium']).sum()), 'of', len(cl))
cl.loc[cl.confidence.isin(['low', 'none', 'medium']),
       ['project', 'title', 'class', 'cue', 'confidence', 'note']]

class
affinity_purification    35
fractionation             2
other                    23
proximity_labelling      10
ptm_enrichment           11
whole_proteome           13

needs a human eye (confidence low/none/medium): 44 of 94


,project,title,class,cue,confidence,note
2,PXD004083,Toxoplasma gondii Methylscan - Comparative Monomethylarginine Proteomics Suggests that PRMT1 is a Significant Contributor to Arginine Monomethylation in Toxoplasma gondii,ptm_enrichment,title: 'Methylscan',medium,title also matches whole_proteome
3,PXD004426,O-fucosylated glycoproteins of Toxoplasma gondii,other,no cue matched,none,no keyword matched the title or the file names
5,PXD006235,Toxoplasma gondii aspartyl protease 3,other,no cue matched,none,no keyword matched the title or the file names
6,PXD007777,iTRAQ-based Quantitative Phosphoproteomics Revealed Differences in Phosphorylation Modification Between Different Toxoplasma gondii Genotypes,ptm_enrichment,title: 'Phospho',medium,title also matches whole_proteome
10,PXD011692,Phosphatidic acid governs natural egress in Toxoplasma gondii via a guanylate cyclase receptor platform,other,no cue matched,none,no keyword matched the title or the file names
12,PXD015809,First Global Proteomic Analysis of Lysine Malonylation in Toxoplasma gondii,ptm_enrichment,title: 'Malonylation',medium,title also matches whole_proteome
15,PXD017032,Comparative phosphoproteomic analysis of sporulated oocysts between virulent and avirulent strains of Toxoplasma gondii reveals strain-specific patterns,ptm_enrichment,title: 'phospho',medium,title also matches whole_proteome
16,PXD018056,Target identification of the novel Toxoplasma gondii rhoptry kinase RON13,affinity_purification,file names: '_IP_',low,title gave no cue; the call rests on deposited file names
17,PXD021516,Strucuture-function relationships of an atypical secretory kinase essential for Toxoplasma gondii invasion,other,no cue matched,none,no keyword matched the title or the file names
19,PXD021938,Mass spectrometry of circulating antigens products isolated from sera of mice infected with Toxoplasma gondii tachyzoites,whole_proteome,title: 'antigen',medium,file names suggest affinity_purification instead


In [8]:
# every project, its class and how it was decided
cl.sort_values(['class', 'project'])[['project', 'class', 'confidence', 'decided_by',
                                      'cue', 'n_runs', 'title']]

,project,class,confidence,decided_by,cue,n_runs,title
8,PXD009799,affinity_purification,high,title,title: 'Immunoprecipit',32,ICAP2 Immunoprecipitation results
11,PXD012103,affinity_purification,high,title,title: 'Immunoprecipit',4,Toxoplasma HCE1 Immunoprecipitations
13,PXD016383,affinity_purification,high,title,title: 'Immunoprecipit',4,Toxoplasma MYR1 Immunoprecipitations
14,PXD016846,affinity_purification,high,title,title: 'interactome',30,Mass spectrometry-based characterization of Toxoplasma gondii MORC interactome
16,PXD018056,affinity_purification,low,file names,file names: '_IP_',32,Target identification of the novel Toxoplasma gondii rhoptry kinase RON13
25,PXD024326,affinity_purification,high,title,title: 'Interactome',56,Interactome of Toxoplasma gondii Pre-mRNA 3’ Processing Complexes
27,PXD027029,affinity_purification,high,title,title: 'interacting prote',24,mGBP2 interacting proteins during Toxoplasma gondii infection
28,PXD028781,affinity_purification,high,title,title: 'interactome',6,Toxoplasma gondii MOB1 interactome
33,PXD032319,affinity_purification,high,title,title: 'Immunoprecipit',5,Toxoplasma gondii PRU ROP1-HA Co-Immunoprecipitation
34,PXD032360,affinity_purification,medium,title,title: 'associated complex',2,MS-based proteomic characterization of Toxoplasma gondii GRA28-associated complexes


## 3 · Which protein was the bait

For the background table itself the bait does not matter: we count how often a protein appears whatever was pulled. The bait matters for two other things. It tells us whether the experiments really are independent (ten pull-downs of the same complex are not ten independent looks at the background), and it is what a worked example needs.

Baits are read out of the title and the run names, then the symbol is resolved to an ME49 gene ID through the ToxoDB product names in `nodes.parquet` plus a short alias table. A bait counts as **confident** only when the symbol resolves to exactly one gene.

The author-reported `interaction_studies.parquet` (97 parsed studies, 31 with a bait) is a separate resource and is reported separately below. It is keyed by PMID and the PRIDE selection carries no PMID, so the two cannot be joined here — the author table is used as a comparison of how hard bait identification is, not as a source of baits for our projects.

In [9]:
nodes = pd.read_parquet(os.path.join(REPO, 'parascape', 'data', 'nodes.parquet'),
                        columns=['gene_id', 'product', 'compartment'])
nodes['product'] = nodes['product'].fillna('')
print('genes:', len(nodes))

def resolve_symbol(sym):
    """ME49 gene IDs whose ToxoDB product name carries this symbol as a whole word."""
    if not sym:
        return []
    pat = r'(?<![A-Za-z0-9])' + re.escape(sym) + r'(?![A-Za-z0-9])'
    hit = nodes[nodes['product'].str.contains(pat, case=False, regex=True)]
    return sorted(hit.gene_id)

print(resolve_symbol('MyoA'), resolve_symbol('MIC2'), resolve_symbol('HDAC3'))

genes: 8140
[] ['TGME49_201780', 'TGME49_214940'] ['TGME49_227290']


In [10]:
# Bait symbol patterns, applied to the title first and the run names second.
BAIT_TITLE_PATTERNS = [
    r'immunoprecipitation(?:\s+mass[- ]spectrometry)?\s+of\s+(?:the\s+)?(?:toxoplasma(?:\s+gondii)?\s+)?([A-Za-z0-9_.-]{2,20})',
    r'(?:toxoplasma(?:\s+gondii)?\s+)?(?:pru\s+)?([A-Za-z0-9_.-]{2,20})[- ]?(?:ha|mng|ty|flag|mng-maid)?\s+(?:co-?)?immunoprecipitation',
    r'([A-Za-z0-9_.-]{2,20})\s+interactome',
    r'([A-Za-z0-9_.-]{2,20})\s+proxisome',
    r'proximity labeling of\s+(?:turboid-)?([A-Za-z0-9_.-]{2,20})',
    r'([A-Za-z0-9_.-]{2,20})-turboid',
    r'turboid-([A-Za-z0-9_.-]{2,20})',
    r'([A-Za-z0-9_.-]{2,20})-bioid2?',
    r'([A-Za-z0-9_.-]{2,20})\s+interactors',
    r'proteins that interact with\s+([A-Za-z0-9_.-]{2,20})',
    r'interacting proteomics of .*\(([A-Za-z0-9]{2,20})\)',
    r'([A-Za-z0-9_.-]{2,20})[- ](?:associated|binding) ',
    r'([A-Za-z0-9_.-]{2,20})\s+associates with',
]
# Epitope and degron tags, stripped before the symbol is looked up.
TAGS = r'(?:[-_ ](?:ha|ty|flag|mng|maid|aid|myc|turboid|bioid2?|bira|gfp|3xha|2xty|toxoplasma|tagged|second|copy))+$'
# Aliases: the name a title uses against a token that really appears in a ToxoDB product
# name. The gene ID is never typed; it is looked up through the product name, and the
# product name is recorded in the output so the mapping can be checked.
ALIAS = {
    'myoa': 'myosin A',          # the product name is 'myosin A', not the symbol
    'hce1': 'TEEGR',             # published synonym; product 'E2F4-associated ... TEEGR'
    'pka c3': 'PKAc3', 'pkac3': 'PKAc3', 'c3': 'PKAc3',
    'calmodulin': 'CAM1',
    'tgbdp1': 'BDP1', 'bdp1': 'BDP1',
    'tgsir2a': 'SIR2A', 'sir2a': 'SIR2A',
    'tggsk3': 'GSK3', 'gsk3': 'GSK3',
    'tgcoqfmo': 'CoqFMO',
    'cyclin-4': 'cyclin4', 'cyclin4': 'cyclin4',
}
# Titles that name a whole complex rather than one tagged protein. The bait is a complex,
# which is a different thing from an unknown bait and is recorded as such.
COMPLEX_BAIT_WORDS = r'succinate dehydrogenase|metc|mitochondrial ribosome|conoid|supercomplex|glycan bead|endocytic and exocytic'
HOST_BAITS = {'mospd2'}  # a host protein, not an ME49 gene

def strip_tags(raw):
    prev = None
    s = raw
    while s != prev:
        prev = s
        s = re.sub(TAGS, '', s, flags=re.I).strip('-_. ')
    return s

STOPWORDS = {'the', 'toxoplasma', 'gondii', 'mass', 'a', 'an', 'protein', 'proteins',
             'and', 'of', 'ms', 'ha', 'anti', 'homolg', 'homolog', 'mitochondrial',
             'succinate', 'novel', 'components', 'this'}

def find_bait(title, names):
    """Bait symbol from the title first, then from the deposited run names."""
    for pat in BAIT_TITLE_PATTERNS:
        m = re.search(pat, title, re.I)
        if not m:
            continue
        raw = strip_tags(m.group(1).strip('-_. '))
        if raw.lower() in STOPWORDS or len(raw) < 2:
            continue
        return raw, 'title'
    # run names: a six-digit ME49 number, or the token before an IP / tag marker
    m = re.search(r'(?<!\d)(\d{6})(?!\d)', names)
    if m and ('TGME49_' + m.group(1)) in set(nodes.gene_id):
        return 'TGME49_' + m.group(1), 'run names (ME49 number in the file name)'
    m = re.search(r'(?:^|[_ ])IP[_-]([A-Za-z0-9]{3,20})', names)
    if m:
        raw = strip_tags(re.sub(r'(AID|HA|Ty|mNG)$', '', m.group(1)))
        if raw.lower() not in STOPWORDS and len(raw) >= 2:
            return raw, 'run names (token after IP_)'
    return '', ''

bait_rows = []
for _, r in cl.iterrows():
    names = ' '.join(det[r.project]['run_names'])
    raw, how = find_bait(r.title, names)
    sym = ALIAS.get(raw.lower(), raw)
    if raw.lower() in HOST_BAITS:
        bait_rows.append(dict(project=r.project, bait_symbol=raw, bait_gene='HOST',
                              bait_confidence='host bait', bait_how=how,
                              bait_product=''))
        continue
    if re.search(COMPLEX_BAIT_WORDS, r.title, re.I):
        bait_rows.append(dict(project=r.project, bait_symbol=raw,
                              bait_gene='', bait_how=how or 'title names a complex',
                              bait_confidence='a complex or a non-protein bait, not one tagged gene', bait_product=''))
        continue
    if re.fullmatch(r'TGME49_\d+', sym):
        genes = [sym]
    else:
        genes = resolve_symbol(sym)
    if len(genes) == 1:
        conf = 'confident'
    elif len(genes) > 1:
        conf = 'ambiguous (symbol matches several genes)'
    elif raw:
        conf = 'ambiguous (symbol not in any ToxoDB product name)'
    else:
        conf = 'none'
    prod = ''
    if len(genes) == 1:
        prod = nodes.loc[nodes.gene_id == genes[0], 'product'].iloc[0]
    bait_rows.append(dict(project=r.project, bait_symbol=raw,
                          bait_gene=genes[0] if len(genes) == 1 else ';'.join(genes),
                          bait_confidence=conf, bait_how=how, bait_product=prod))
bt = pd.DataFrame(bait_rows)
# Baits resolved from the depositing papers (instruction 80): each row names its source and
# the sentence that identifies the tagged gene. A symbol match alone, or no paper, stays
# ambiguous rather than being guessed. Projects the title rule left as 'other' are
# reclassified where the paper shows an affinity purification.
cur = pd.read_csv(os.path.join(REPO, 'parascape', 'data', 'pulldown_baits_curated.tsv'),
                  sep='\t', dtype=str).fillna('')
for _, c in cur.iterrows():
    conf = ('confident' if c['confidence'] in ('paper', 'ToxoDB gene name')
            else 'ambiguous (' + c['confidence'] + ')')
    row = dict(project=c['project'], bait_symbol=c['bait_symbol'], bait_gene=c['bait_gene'],
               bait_confidence=conf, bait_how='curated: ' + c['source'], bait_product='')
    bt = pd.concat([bt[bt.project != c['project']], pd.DataFrame([row])], ignore_index=True)
    cl.loc[cl.project == c['project'], 'class'] = c['class']
print('baits resolved from papers or ToxoDB names:',
      int((cur.confidence.isin(['paper', 'ToxoDB gene name'])).sum()), 'of', len(cur))
cl = cl.merge(bt, on='project', how='left')
pull = cl[cl['class'].isin(['affinity_purification', 'proximity_labelling'])]
print('affinity + proximity projects:', len(pull))
print(pull.bait_confidence.value_counts().to_string())
print()
print('distinct confident bait genes:',
      pull.loc[pull.bait_confidence == 'confident', 'bait_gene'].nunique())
pull[['project', 'class', 'bait_symbol', 'bait_gene', 'bait_confidence', 'title']]

baits resolved from papers or ToxoDB names: 12 of 15
affinity + proximity projects: 50
bait_confidence
confident                                                              29
none                                                                    8
a complex or a non-protein bait, not one tagged gene                    6
ambiguous (symbol not in any ToxoDB product name)                       2
ambiguous (unresolved)                                                  1
host bait                                                               1
ambiguous (symbol matches several genes)                                1
ambiguous (paper names the gene; the pull-down itself not verified)     1
ambiguous (symbol match only (paper not readable))                      1

distinct confident bait genes: 25


,project,class,bait_symbol,bait_gene,bait_confidence,title
8,PXD009799,affinity_purification,ICAP2,,ambiguous (symbol not in any ToxoDB product name),ICAP2 Immunoprecipitation results
10,PXD011692,affinity_purification,(not stated),,ambiguous (unresolved),Phosphatidic acid governs natural egress in Toxoplasma gondii via a guanylate cyclase receptor platform
11,PXD012103,affinity_purification,HCE1,TGME49_239010,confident,Toxoplasma HCE1 Immunoprecipitations
13,PXD016383,affinity_purification,MYR1,TGME49_254470,confident,Toxoplasma MYR1 Immunoprecipitations
14,PXD016846,affinity_purification,MORC,TGME49_305340,confident,Mass spectrometry-based characterization of Toxoplasma gondii MORC interactome
16,PXD018056,affinity_purification,,,none,Target identification of the novel Toxoplasma gondii rhoptry kinase RON13
22,PXD022785,proximity_labelling,,,"a complex or a non-protein bait, not one tagged gene",Toxoplasma conoid proteome by BioID
25,PXD024326,affinity_purification,,,none,Interactome of Toxoplasma gondii Pre-mRNA 3’ Processing Complexes
27,PXD027029,affinity_purification,,,none,mGBP2 interacting proteins during Toxoplasma gondii infection
28,PXD028781,affinity_purification,MOB1,,ambiguous (symbol not in any ToxoDB product name),Toxoplasma gondii MOB1 interactome


In [11]:
# the author-reported table, for comparison only
st = pd.read_parquet(os.path.join(REPO, 'parascape', 'data', 'interaction_studies.parquet'))
mem = pd.read_parquet(os.path.join(REPO, 'parascape', 'data',
                                   'interaction_study_members.parquet'))
print('parsed interaction studies:', len(st),
      '| with a bait gene:', int((st.bait_gene.fillna('') != '').sum()))
print('bait-member pairs:', len(mem))
print('methods:'); print(st.method.value_counts().to_string())
print()
print('can these be joined to our PXD projects?  columns available:', list(st.columns))
print('-> no accession column, so no join; used as a comparison only.')
st.method.value_counts()

parsed interaction studies: 97 | with a bait gene: 31
bait-member pairs: 68362
methods:
method
IPMS     55
BioID    42

can these be joined to our PXD projects?  columns available: ['pmid', 'pmcid', 'year', 'method', 'title', 'n_files', 'n_tables', 'n_genes', 'parsed', 'unreadable_formats', 'bait_gene', 'bait_confidence']
-> no accession column, so no join; used as a comparison only.


method
IPMS     55
BioID    42

## 4 · The detection rule

Sage reports, for every peptide-spectrum match, a `spectrum_q` (the false-discovery rate at which that match would be accepted) and the protein or proteins the peptide belongs to. The rule used for the background table is:

> **A protein is detected in an experiment when at least 2 peptide-spectrum matches at 1 % spectrum-level FDR map to that gene and to no other parasite gene.**

Three choices, each for a reason:

* **1 % FDR** is the field standard and it is Sage's own number, so no new threshold is invented here.

* **At least 2 matches** removes single-spectrum identifications. At 1 % FDR a single match is the one most likely to be wrong, and for a background table a false detection is worse than a missed one: it would make a protein look like a frequent flyer and get a real hit filtered out of someone's list.

* **Unique mapping.** If a peptide is shared between two paralogues, crediting both would inflate the frequency of whole families (SRS surface antigens, the myosins). A protein seen only through shared peptides is not counted — a conservative choice that makes frequencies slightly too low rather than too high.

Section 6 repeats the whole table under a looser rule (1 match) and a stricter one (at least 2 distinct peptide sequences and at least 3 matches), so the sensitivity of every conclusion to this choice is visible.

**Experiments, not runs.** One project is one experiment. A project with eight runs is usually the bait and its control in replicate, which is one look at the background, not eight. Run-level counts are also recorded so the difference can be seen.

In [12]:
GENE = re.compile(r'TGME49_\d+')
PSM_COLS = ['peptide', 'proteins', 'label', 'spectrum_q', 'ms2_intensity', 'filename']

def aggregate_psms(path, chunksize=400_000):
    """One pass over a project's Sage PSM table. Returns a per-gene frame and totals.

    Only the aggregate is kept in memory, so the table size does not matter."""
    parts, n_rows, n_pass = [], 0, 0
    for ch in pd.read_csv(path, sep='\t', usecols=PSM_COLS, chunksize=chunksize):
        n_rows += len(ch)
        ch = ch[(ch['label'] == 1) & (ch['spectrum_q'] <= 0.01)]
        n_pass += len(ch)
        if not len(ch):
            continue
        genes = ch['proteins'].astype(str).map(lambda s: set(GENE.findall(s)))
        one = genes.map(len) == 1
        ch = ch.loc[one].copy()
        if not len(ch):
            continue
        ch['gene_id'] = [next(iter(g)) for g in genes.loc[one]]
        g = ch.groupby('gene_id').agg(
            n_psm=('peptide', 'size'),
            n_peptides=('peptide', 'nunique'),
            n_runs=('filename', 'nunique'),
            intensity=('ms2_intensity', 'sum'))
        pep = ch.groupby('gene_id')['peptide'].apply(lambda s: set(s))
        run = ch.groupby('gene_id')['filename'].apply(lambda s: set(s))
        g['peptide_set'] = pep
        g['run_set'] = run
        parts.append(g)
    empty = pd.DataFrame({'gene_id': pd.Series(dtype=str),
                          'n_psm': pd.Series(dtype=int), 'intensity': pd.Series(dtype=float),
                          'n_peptides': pd.Series(dtype=int),
                          'n_runs_detected': pd.Series(dtype=int)})
    if not parts:
        return empty, dict(n_rows=n_rows, n_pass=n_pass)
    cat = pd.concat(parts)
    out = cat.groupby(level=0).agg(
        n_psm=('n_psm', 'sum'), intensity=('intensity', 'sum'),
        peptide_set=('peptide_set', lambda ss: set().union(*ss)),
        run_set=('run_set', lambda ss: set().union(*ss)))
    out['n_peptides'] = out.peptide_set.map(len)
    out['n_runs_detected'] = out.run_set.map(len)
    out = out.drop(columns=['peptide_set', 'run_set']).reset_index(names='gene_id')
    return out, dict(n_rows=n_rows, n_pass=n_pass)

LFQ_META = {'peptide', 'charge', 'proteins', 'q_value', 'score', 'spectral_angle'}

def aggregate_lfq(path):
    """Per-gene MS1 LFQ intensity, summed over the project's runs."""
    t = pd.read_csv(path, sep='\t')
    runs = [c for c in t.columns if c not in LFQ_META]
    t = t[t['q_value'] <= 0.01]
    genes = t['proteins'].astype(str).map(lambda s: set(GENE.findall(s)))
    one = genes.map(len) == 1
    t = t.loc[one].copy()
    if not len(t):
        return pd.DataFrame(columns=['gene_id', 'lfq'])
    t['gene_id'] = [next(iter(g)) for g in genes.loc[one]]
    t['lfq'] = t[runs].sum(axis=1)
    return t.groupby('gene_id', as_index=False)['lfq'].sum()
print('aggregators defined')

aggregators defined


In [13]:
# one project at a time; only the aggregate is kept
per_project, psm_stats = {}, []
for _, r in cl.iterrows():
    row = usable.loc[usable.project == r.project].iloc[0]
    agg, stats_ = aggregate_psms(row.psm_path)
    if row.lfq_path:
        lfq = aggregate_lfq(row.lfq_path)
        agg = agg.merge(lfq, on='gene_id', how='left')
    else:
        agg['lfq'] = np.nan
    agg['project'] = r.project
    per_project[r.project] = agg
    psm_stats.append(dict(project=r.project, psm_rows=stats_['n_rows'],
                          psm_1pct=stats_['n_pass'],
                          genes_1psm=int((agg.n_psm >= 1).sum()),
                          genes_2psm=int((agg.n_psm >= 2).sum()),
                          has_lfq=bool(row.lfq_path),
                          lfq_genes=int(agg.lfq.notna().sum())))
ps = pd.DataFrame(psm_stats)
cl = cl.merge(ps, on='project', how='left')
print('projects aggregated:', len(per_project))
print('PSM rows read:', int(ps.psm_rows.sum()),
      '| passing 1% FDR and target:', int(ps.psm_1pct.sum()))
ps.describe()[['psm_rows', 'psm_1pct', 'genes_1psm', 'genes_2psm']]

projects aggregated: 94
PSM rows read: 15956399 | passing 1% FDR and target: 3975917


,psm_rows,psm_1pct,genes_1psm,genes_2psm
count,94,94,94,94
mean,1.697e+05,4.23e+04,1126,862.4
std,1.297e+05,4.557e+04,898.4,748.2
min,6258,0,0,0
25%,6.206e+04,1.009e+04,346.5,250.5
50%,1.384e+05,2.751e+04,1018,750.5
75%,2.313e+05,6.114e+04,1602,1255
max,5.928e+05,2.785e+05,4210,3592


## 5 · The background table

Only the affinity-purification and proximity-labelling projects contribute. Abundance is made comparable between projects by expressing each protein as parts per million of that project's own total parasite signal, so a deep run and a shallow run can be averaged. Two abundance scales are kept: MS1 label-free quantity (`lfq_ppm`, the better measure, available where the pipeline wrote an `lfq.tsv`) and summed MS2 fragment intensity of the counted matches (`psm_ppm`, available everywhere, including the timsTOF projects which have no MS1 LFQ).

In [14]:
BG_CLASSES = ['affinity_purification', 'proximity_labelling']
bg_projects = sorted(cl.loc[cl['class'].isin(BG_CLASSES), 'project'])
N_EXP = len(bg_projects)
print('experiments in the background:', N_EXP)
print(cl[cl['class'].isin(BG_CLASSES)].groupby('class').size().to_string())

frames = []
for p in bg_projects:
    a = per_project[p].copy()
    tot_i = a.intensity.sum()
    tot_l = a.lfq.sum(skipna=True)
    a['psm_ppm'] = a.intensity / tot_i * 1e6 if tot_i > 0 else np.nan
    a['lfq_ppm'] = a.lfq / tot_l * 1e6 if tot_l and tot_l > 0 else np.nan
    frames.append(a)
long = pd.concat(frames, ignore_index=True)
print('gene x experiment rows (>=1 unique match):', len(long))
long.head(5)

experiments in the background: 50
class
affinity_purification    40
proximity_labelling      10
gene x experiment rows (>=1 unique match): 59815


,gene_id,n_psm,intensity,n_peptides,n_runs_detected,lfq,project,psm_ppm,lfq_ppm
0,TGME49_201210,1,7267,1,1,NaN,PXD009799,4.213,NaN
1,TGME49_201800,299,3.093e+07,15,22,3.897e+07,PXD009799,1.793e+04,8380
2,TGME49_201840,1,9392,1,1,NaN,PXD009799,5.445,NaN
3,TGME49_202680,7,1.479e+05,3,3,NaN,PXD009799,85.77,NaN
4,TGME49_203240,1,2589,1,1,NaN,PXD009799,1.501,NaN


In [15]:
def build_background(long, n_exp, min_psm=2, min_peptides=1):
    d = long[(long.n_psm >= min_psm) & (long.n_peptides >= min_peptides)]
    g = d.groupby('gene_id').agg(
        experiments_detected=('project', 'nunique'),
        runs_detected=('n_runs_detected', 'sum'),
        median_psm_ppm=('psm_ppm', 'median'), max_psm_ppm=('psm_ppm', 'max'),
        median_abundance=('lfq_ppm', 'median'), max_abundance=('lfq_ppm', 'max'),
        n_experiments_with_lfq=('lfq_ppm', 'count'),
        total_psm=('n_psm', 'sum'))
    g['n_experiments'] = n_exp
    g['frequency'] = g.experiments_detected / n_exp
    return g.reset_index()

bg = build_background(long, N_EXP)
bg = bg.merge(nodes.rename(columns={'product': 'product_name'}), on='gene_id', how='left')
print('proteins in the background table:', len(bg))
print('detected in every experiment:', int((bg.frequency == 1).sum()))
print('detected in >= 80% of experiments:', int((bg.frequency >= 0.8).sum()))
print('detected in exactly one experiment:', int((bg.experiments_detected == 1).sum()))
print()
print('frequency distribution:')
print(pd.cut(bg.frequency, [0, .1, .25, .5, .75, .9, 1.0],
             include_lowest=True).value_counts().sort_index().to_string())
bg.sort_values('frequency', ascending=False).head(25)[
    ['gene_id', 'product_name', 'experiments_detected', 'frequency',
     'median_abundance', 'compartment']]

proteins in the background table: 4952
detected in every experiment: 0
detected in >= 80% of experiments: 90
detected in exactly one experiment: 832

frequency distribution:
frequency
(-0.001, 0.1]    2493
(0.1, 0.25]      1160
(0.25, 0.5]       840
(0.5, 0.75]       337
(0.75, 0.9]       109
(0.9, 1.0]         13


,gene_id,product_name,experiments_detected,frequency,median_abundance,compartment
338,TGME49_209030,actin ACT1,48,0.96,1.057e+04,cytosol
4682,TGME49_316400,alpha tubulin TUBA1,48,0.96,5947,tubulin cytoskeleton
4418,TGME49_311720,chaperonin protein BiP,47,0.94,6788,ER 2
3248,TGME49_273760,heat shock protein HSP70,47,0.94,4075,cytosol
1746,TGME49_239260,histone H4,47,0.94,3526,nucleus - chromatin
160,TGME49_204020,ribosomal protein RPL8,47,0.94,3544,60S ribosome
1242,TGME49_227360,ribosomal protein RPL3,47,0.94,3032,60S ribosome
1224,TGME49_226970,ribosomal protein RPS11,46,0.92,2400,40S ribosome
177,TGME49_204400,"ATPase synthase subunit alpha, putative",46,0.92,1863,mitochondrion - membranes
4253,TGME49_308090,rhoptry protein ROP5,46,0.92,1.542e+04,rhoptries 1


### Runs are not experiments

The same table counted by run instead of by project shows why the unit matters: the run count is dominated by whichever projects happened to deposit many files.

In [16]:
n_runs_total = int(cl.loc[cl['class'].isin(BG_CLASSES), 'n_runs'].sum())
cmp = bg[['gene_id', 'product_name', 'experiments_detected', 'frequency', 'runs_detected']].copy()
cmp['run_frequency'] = cmp.runs_detected / n_runs_total
print('runs behind the', N_EXP, 'experiments:', n_runs_total)
print('rank correlation of the two frequencies: %.3f' %
      stats.spearmanr(cmp.frequency, cmp.run_frequency).statistic)
print('genes in the top 50 by experiment frequency but not by run frequency:',
      len(set(cmp.nlargest(50, 'frequency').gene_id) -
          set(cmp.nlargest(50, 'run_frequency').gene_id)))
cmp.nlargest(10, 'run_frequency')[['gene_id', 'product_name', 'frequency', 'run_frequency']]

runs behind the 50 experiments: 440
rank correlation of the two frequencies: 0.968
genes in the top 50 by experiment frequency but not by run frequency: 10


,gene_id,product_name,frequency,run_frequency
338,TGME49_209030,actin ACT1,0.96,0.7727
4682,TGME49_316400,alpha tubulin TUBA1,0.96,0.7295
3248,TGME49_273760,heat shock protein HSP70,0.94,0.725
4253,TGME49_308090,rhoptry protein ROP5,0.92,0.6659
4418,TGME49_311720,chaperonin protein BiP,0.94,0.6568
177,TGME49_204400,"ATPase synthase subunit alpha, putative",0.92,0.6432
2225,TGME49_250770,"eukaryotic initiation factor-4A, putative",0.86,0.6432
1746,TGME49_239260,histone H4,0.94,0.6295
2683,TGME49_261950,ATP synthase beta subunit ATP-B,0.88,0.6227
2792,TGME49_263700,ribosomal protein RPS14,0.92,0.6227


## 6 · How much the detection rule matters

The same table under a looser and a stricter rule. If the ranking barely moves, the threshold is not what the conclusions rest on.

In [17]:
variants = {
    'loose (>=1 match)': build_background(long, N_EXP, min_psm=1),
    'default (>=2 matches)': bg[['gene_id', 'experiments_detected', 'frequency']].copy(),
    'strict (>=3 matches, >=2 peptides)': build_background(long, N_EXP, min_psm=3,
                                                          min_peptides=2),
}
rows = []
ref = variants['default (>=2 matches)'].set_index('gene_id').frequency
for name, v in variants.items():
    f = v.set_index('gene_id').frequency
    both = ref.index.intersection(f.index)
    rows.append(dict(rule=name, proteins=len(v),
                     freq_ge_0_5=int((f >= 0.5).sum()),
                     freq_ge_0_8=int((f >= 0.8).sum()),
                     seen_once=int((v.set_index('gene_id').experiments_detected == 1).sum()),
                     median_frequency=round(float(f.median()), 4),
                     spearman_vs_default=round(float(
                         stats.spearmanr(ref[both], f[both]).statistic), 4)))
sens = pd.DataFrame(rows)
sens

,rule,proteins,freq_ge_0_5,freq_ge_0_8,seen_once,median_frequency,spearman_vs_default
0,loose (>=1 match),5775,683,127,807,0.14,0.963
1,default (>=2 matches),4952,495,90,832,0.1,1
2,"strict (>=3 matches, >=2 peptides)",4290,385,62,774,0.1,0.9635


## 7 · The specificity score

Given a hit list from one experiment — a protein and how much of it there was — the score asks two questions and adds the answers:

1. **Is there more of it here than usual?** `log2` of this experiment's parts per million against the median parts per million the protein reaches across the background experiments in which it is seen at all.

2. **Is it unusual to see it at all?** `log2` of one over its background frequency, smoothed so a protein seen in no background experiment does not give an infinite score.

`specificity = log2(abundance fold) + log2(1 / frequency)`. A protein seen in every pull-down at its usual level scores about 0. Tubulin at twice its usual level still scores low, because the second term is near zero. A protein never seen before, at a decent level, scores high.

Both parts are reported separately, because they fail differently: the fold change is unreliable for a protein with few background measurements, and the frequency is unreliable for everything while the number of experiments is small.

In [18]:
def specificity(hits, background, n_experiments=None, pseudo=0.5, abundance='psm_ppm'):
    """Score a pull-down hit list against the background table.

    hits        : DataFrame with 'gene_id' and an abundance column in parts per million
                  of that experiment's own total parasite signal.
    background  : the background table (gene_id, experiments_detected, n_experiments,
                  frequency, median_psm_ppm / median_abundance).
    Returns the hit list with fold, frequency and the score, highest score first.
    """
    bgcol = 'median_abundance' if abundance == 'lfq_ppm' else 'median_psm_ppm'
    n = int(background.n_experiments.iloc[0]) if n_experiments is None else n_experiments
    b = background[['gene_id', 'experiments_detected', bgcol]].rename(
        columns={bgcol: 'background_ppm'})
    h = hits.merge(b, on='gene_id', how='left')
    h['experiments_detected'] = h.experiments_detected.fillna(0)
    h['n_experiments'] = n
    # smoothed frequency: never 0, never exactly 1
    h['background_frequency'] = (h.experiments_detected + pseudo) / (n + pseudo)
    typical = float(background[bgcol].median(skipna=True))
    bgp = h.background_ppm.fillna(typical)
    h['abundance_fold'] = (h[abundance] + 1.0) / (bgp + 1.0)
    h['log2_fold'] = np.log2(h.abundance_fold)
    h['log2_rarity'] = np.log2(1.0 / h.background_frequency)
    h['specificity'] = h.log2_fold + h.log2_rarity
    return h.sort_values('specificity', ascending=False).reset_index(drop=True)
print(specificity.__doc__.splitlines()[0])

Score a pull-down hit list against the background table.


### A worked example: the MYR1 immunoprecipitation (PXD016383)

MYR1 is the translocon component that moves dense granule effectors across the parasitophorous vacuole membrane into the host cell. Its partners are known from the literature — MYR2, MYR3, MYR4, GRA44, GRA45, ROP17 — so this is a hit list whose right answer we already have. The project itself contributes to the background, which makes the test slightly harder, not easier.

In [19]:
EXAMPLE = 'PXD016383'
ex = per_project[EXAMPLE].copy()
tot = ex.intensity.sum()
ex['psm_ppm'] = ex.intensity / tot * 1e6
ex = ex[ex.n_psm >= 2]
scored = specificity(ex[['gene_id', 'n_psm', 'n_peptides', 'psm_ppm']], bg)
scored = scored.merge(nodes.rename(columns={'product': 'product_name'}),
                      on='gene_id', how='left')
print(cl.loc[cl.project == EXAMPLE, ['title', 'class', 'bait_symbol', 'bait_gene']].to_string(index=False))
print('proteins in the hit list:', len(scored))
scored.head(25)[['gene_id', 'product_name', 'n_psm', 'psm_ppm', 'background_frequency',
                 'log2_fold', 'log2_rarity', 'specificity']]

                               title                 class bait_symbol     bait_gene
Toxoplasma MYR1 Immunoprecipitations affinity_purification        MYR1 TGME49_254470
proteins in the hit list: 363


,gene_id,product_name,n_psm,psm_ppm,background_frequency,log2_fold,log2_rarity,specificity
0,TGME49_254470,Myc-regulation protein MYR1,285,9.51e+04,0.5248,9.902,0.9303,10.83
1,TGME49_211460,Myc-regulation protein MYR4,66,7603,0.2673,7.288,1.903,9.191
2,TGME49_263560,hypothetical protein,10,1067,0.1287,5.943,2.958,8.901
3,TGME49_500284,"hypothetical protein, conserved",39,1.049e+04,0.4059,6.87,1.301,8.17
4,TGME49_220330,hypothetical protein,10,340.3,0.06931,3.908,3.851,7.759
5,TGME49_248140,hypothetical protein,7,777.5,0.06931,3.578,3.851,7.429
6,TGME49_319340,dense granule protein GRA52,55,1.447e+04,0.5446,6.278,0.8769,7.155
7,TGME49_203600,dense granule protein GRA50,24,5058,0.4851,5.803,1.044,6.847
8,TGME49_204340,dense granule protein GRA54,63,1.685e+04,0.6634,6.252,0.5921,6.844
9,TGME49_233870,hypothetical protein,29,1714,0.2277,4.668,2.135,6.803


In [20]:
# where the known MYR1 complex lands in that ranking
MYR_SYMS = ['MYR1', 'MYR2', 'MYR3', 'MYR4', 'GRA44', 'GRA45', 'ROP17']
want = {}
for s in MYR_SYMS:
    for g in resolve_symbol(s):
        want[g] = s
scored['rank'] = np.arange(1, len(scored) + 1)
scored['percentile'] = 100 * (1 - (scored['rank'] - 1) / len(scored))
known = scored[scored.gene_id.isin(want)].copy()
known['symbol'] = known.gene_id.map(want)
known[['symbol', 'gene_id', 'product_name', 'rank', 'percentile', 'psm_ppm',
       'background_frequency', 'specificity']]

,symbol,gene_id,product_name,rank,percentile,psm_ppm,background_frequency,specificity
0,MYR1,TGME49_254470,Myc-regulation protein MYR1,1,100,9.51e+04,0.5248,10.83
1,MYR4,TGME49_211460,Myc-regulation protein MYR4,2,99.72,7603,0.2673,9.191
18,MYR3,TGME49_237230,membrane protein MYR3,19,95.04,872.5,0.3267,5.976
68,GRA44,TGME49_228170,dense granule protein GRA44,69,81.27,1.146e+04,0.7228,4.812
72,GRA45,TGME49_316250,dense granule protein GRA45,73,80.17,920.3,0.4455,4.642
187,ROP17,TGME49_258580,rhoptry protein ROP17,188,48.48,3564,0.8218,1.797


## 8 · Validation: do real complexes survive, and are the frequent flyers caught?

This is what decides whether the table can be used. Two sets of proteins with a known answer:

* **Known complexes** must have a *low* background frequency. If the glideosome is detected in every pull-down, the table would strip it out of a glideosome paper.

* **Frequent flyers** — ribosomal proteins, HSP70/HSP90, tubulin, actin, GAPDH, enolase — must have a *high* background frequency. If they do not, the table is not catching the thing it exists to catch.

Gene identities come from ToxoDB product names in `nodes.parquet`, listed in full so they can be checked.

In [21]:
# Each complex member is named by the exact ToxoDB product name, so there is no doubt
# which gene is meant and no gene ID is typed from memory.
COMPLEXES = {
    'MyoA glideosome': {'MyoA': 'myosin A', 'MLC1': 'myosin light chain MLC1',
                        'ELC1': 'essential light chain ELC1',
                        'GAP45': 'gliding-associated protein GAP45',
                        'GAP50': 'gliding-associated protein GAP50',
                        'GAP40': 'GAP40 protein'},
    'AMA1-RON':        {'AMA1': 'apical membrane antigen AMA1',
                        'RON2': 'rhoptry neck protein RON2',
                        'RON4': 'rhoptry neck protein RON4'},
    'MIC2-M2AP':       {'MIC2': 'microneme protein MIC2',
                        'M2AP': 'MIC2-associated protein M2AP'},
    'MORC/HDAC3':      {'MORC': 'ATPase MORC',
                        'HDAC3': 'histone deacetylase HDAC3'},
}
# Frequent flyers, tight: the actual abundant housekeeping proteins. The cytosolic
# ribosome is taken from the curated hyperLOPIT 40S/60S markers rather than from a product
# name, so the apicoplast and mitochondrial 30S/50S ribosomes are not mixed in.
FLYER_TIGHT = {
    'HSP70/HSP90': r'^heat shock protein (?:hsp)?(?:70|90)(?:,|$)',
    'tubulin':     r'^(?:(?:alpha|beta|beta-1|gamma)[- ]tubulin|tubulin (?:gamma|alpha|beta) chain)(?!.*(?:ligase|suppressor|transferase|cofactor|complex component|binding))',
    'actin':       r'^actin ACT1$',
    'GAPDH':       r'^glyceraldehyde-3-phosphate dehydrogenase GAPDH',
    'enolase':     r'^enolase \d$',
}
# Frequent flyers, loose: whole families as a keyword would find them. Reported beside the
# tight sets because the loose sets sweep in low-abundance relatives (the tubulin-tyrosine
# ligases, the organellar ribosomes) that are not frequent flyers at all.
FLYER_LOOSE = {
    'ribosomal protein (any)': r'ribosomal protein',
    'heat shock (any)':        r'heat shock|hsp\d',
    'tubulin (any)':           r'tubulin',
    'actin (any)':             r'^actin',
}

val_rows = []
for name, members in COMPLEXES.items():
    for sym, product in members.items():
        hit = nodes[nodes['product'].str.lower() == product.lower()]
        if not len(hit):
            print('WARNING: no gene with product %r' % product)
        for g in hit.gene_id:
            val_rows.append(dict(set_name=name, kind='known complex', symbol=sym,
                                 gene_id=g))
for g in nodes.loc[nodes.compartment.isin(['40S ribosome', '60S ribosome']), 'gene_id']:
    val_rows.append(dict(set_name='cytosolic ribosome (hyperLOPIT marker)',
                         kind='expected frequent flyer', symbol='ribosome', gene_id=g))
for name, pat in FLYER_TIGHT.items():
    hit = nodes[nodes['product'].str.contains(pat, case=False, regex=True)]
    for g in hit.gene_id:
        val_rows.append(dict(set_name=name, kind='expected frequent flyer',
                             symbol=name, gene_id=g))
for name, pat in FLYER_LOOSE.items():
    hit = nodes[nodes['product'].str.contains(pat, case=False, regex=True)]
    for g in hit.gene_id:
        val_rows.append(dict(set_name=name, kind='loose family (comparison only)',
                             symbol=name, gene_id=g))
val = pd.DataFrame(val_rows).drop_duplicates(['set_name', 'gene_id'])
val = val.merge(bg[['gene_id', 'product_name', 'experiments_detected', 'n_experiments',
                    'frequency', 'median_abundance', 'median_psm_ppm', 'compartment']],
                on='gene_id', how='left')
val['experiments_detected'] = val.experiments_detected.fillna(0).astype(int)
val['n_experiments'] = N_EXP
val['frequency'] = val.frequency.fillna(0.0)
print('validation proteins:', len(val))
val.groupby(['kind', 'set_name']).agg(
    proteins=('gene_id', 'size'),
    median_frequency=('frequency', 'median'),
    max_frequency=('frequency', 'max'),
    never_seen=('experiments_detected', lambda s: int((s == 0).sum())))

validation proteins: 311


proteins  median_frequency  max_frequency  never_seen
kind                           set_name                                                                                     
expected frequent flyer        GAPDH                                          2              0.81           0.84           0
                               HSP70/HSP90                                    3              0.88           0.94           0
                               actin                                          1              0.96           0.96           0
                               cytosolic ribosome (hyperLOPIT marker)        68              0.80           0.94           0
                               enolase                                        2              0.25           0.50           1
                               tubulin                                        6              0.61           0.96           2
known complex                  AMA1-RON                                       3              0.82           0.84           0
                               MIC2-M2AP                                      2              0.59           0.60           0
                               MORC/HDAC3                                     2              0.60           0.70           0
                               MyoA glideosome                                6              0.69           0.86           0
loose family (comparison only) actin (any)                                   12              0.29           0.96           1
                               heat shock (any)                              23              0.24           0.94           5
                               ribosomal protein (any)                      158              0.23           0.94          20
                               tubulin (any)                                 23              0.02           0.96           9

In [22]:
# the named complex members one by one
val[val.kind == 'known complex'].sort_values(['set_name', 'symbol'])[
    ['set_name', 'symbol', 'gene_id', 'product_name', 'experiments_detected',
     'n_experiments', 'frequency', 'median_abundance']]

,set_name,symbol,gene_id,product_name,experiments_detected,n_experiments,frequency,median_abundance
6,AMA1-RON,AMA1,TGME49_255260,apical membrane antigen AMA1,38,50,0.76,614.1
7,AMA1-RON,RON2,TGME49_300100,rhoptry neck protein RON2,41,50,0.82,1483
8,AMA1-RON,RON4,TGME49_229010,rhoptry neck protein RON4,42,50,0.84,1562
10,MIC2-M2AP,M2AP,TGME49_214940,MIC2-associated protein M2AP,29,50,0.58,185.7
9,MIC2-M2AP,MIC2,TGME49_201780,microneme protein MIC2,30,50,0.6,210.6
12,MORC/HDAC3,HDAC3,TGME49_227290,histone deacetylase HDAC3,25,50,0.5,42.92
11,MORC/HDAC3,MORC,TGME49_305340,ATPase MORC,35,50,0.7,112.9
2,MyoA glideosome,ELC1,TGME49_269442,essential light chain ELC1,20,50,0.4,56.92
5,MyoA glideosome,GAP40,TGME49_249850,GAP40 protein,30,50,0.6,186.5
3,MyoA glideosome,GAP45,TGME49_223940,gliding-associated protein GAP45,33,50,0.66,268.1


In [23]:
a = val.loc[val.kind == 'known complex', 'frequency']
b = val.loc[val.kind == 'expected frequent flyer', 'frequency']
u = stats.mannwhitneyu(b, a, alternative='greater')
auc = u.statistic / (len(a) * len(b))
print('known complex members:        n=%d  median frequency %.3f' % (len(a), a.median()))
print('expected frequent flyers:     n=%d  median frequency %.3f' % (len(b), b.median()))
print('all proteins in the table:    n=%d  median frequency %.3f'
      % (len(bg), bg.frequency.median()))
print()
print('flyers above complex members: AUC %.3f, Mann-Whitney p = %.3g' % (auc, u.pvalue))
print('a flyer picked at random beats a complex member %.0f%% of the time' % (100 * auc))
print()
thresh = 0.5
print('at a frequency cutoff of %.0f%%:' % (100 * thresh))
print('  complex members that would be filtered out as background: %d of %d'
      % (int((a >= thresh).sum()), len(a)))
print('  frequent flyers correctly filtered:                      %d of %d'
      % (int((b >= thresh).sum()), len(b)))
pd.DataFrame({'known complex': a.describe(), 'frequent flyer': b.describe(),
              'all proteins': bg.frequency.describe()})

known complex members:        n=13  median frequency 0.700
expected frequent flyers:     n=82  median frequency 0.800
all proteins in the table:    n=4952  median frequency 0.100

flyers above complex members: AUC 0.689, Mann-Whitney p = 0.0145
a flyer picked at random beats a complex member 69% of the time

at a frequency cutoff of 50%:
  complex members that would be filtered out as background: 12 of 13
  frequent flyers correctly filtered:                      74 of 82


,known complex,frequent flyer,all proteins
count,13,82,4952
mean,0.68,0.7463,0.1871
std,0.1398,0.2011,0.1974
min,0.4,0,0.02
25%,0.6,0.74,0.04
50%,0.7,0.8,0.1
75%,0.8,0.84,0.26
max,0.86,0.96,0.96


### Could the frequency be used as a filter at any cutoff?

A direction that is right on average is not the same as a filter that works. The question a biologist actually asks is: if I delete everything above some frequency, how many frequent flyers do I lose and how much of my complex do I lose with them? The sweep below answers that at every cutoff.

In [24]:
sweep = []
for c in np.arange(0.05, 1.001, 0.05):
    sweep.append(dict(cutoff=round(float(c), 2),
                      flyers_removed=int((b >= c).sum()), flyers=len(b),
                      flyer_recall=float((b >= c).mean()),
                      complex_lost=int((a >= c).sum()), complex_members=len(a),
                      complex_loss=float((a >= c).mean()),
                      proteins_removed=int((bg.frequency >= c).sum())))
sw = pd.DataFrame(sweep)
ok = sw[(sw.complex_loss <= 0.15) & (sw.flyer_recall >= 0.5)]
FREQ_FILTER_WORKS = bool(len(ok))
if FREQ_FILTER_WORKS:
    print('a usable cutoff exists:')
    print(ok.to_string(index=False))
else:
    best = sw[sw.flyer_recall >= 0.5].nsmallest(1, 'complex_loss')
    print('NO cutoff removes at least half the frequent flyers while losing at most 15% of')
    print('the known complex members. The least bad such cutoff:')
    print(best.to_string(index=False))
    print()
    print('-> the frequency column must not be used as a standalone filter.')
sw

NO cutoff removes at least half the frequent flyers while losing at most 15% of
the known complex members. The least bad such cutoff:
 cutoff  flyers_removed  flyers  flyer_recall  complex_lost  complex_members  complex_loss  proteins_removed
    0.8              50      82      0.609756             4               13      0.307692                90

-> the frequency column must not be used as a standalone filter.


,cutoff,flyers_removed,flyers,flyer_recall,complex_lost,complex_members,complex_loss,proteins_removed
0,0.05,79,82,0.9634,13,13,1,3555
1,0.1,79,82,0.9634,13,13,1,2746
2,0.15,79,82,0.9634,13,13,1,2010
3,0.2,79,82,0.9634,13,13,1,1655
4,0.25,79,82,0.9634,13,13,1,1299
5,0.3,78,82,0.9512,13,13,1,1093
6,0.35,77,82,0.939,13,13,1,848
7,0.4,75,82,0.9146,13,13,1,727
8,0.45,75,82,0.9146,12,13,0.9231,587
9,0.5,74,82,0.9024,12,13,0.9231,495


### The loose family sets, for comparison

The same test with the frequent-flyer sets defined by a keyword instead of curated. The loose sets sweep in the tubulin-tyrosine ligases and the apicoplast and mitochondrial 30S/50S ribosomes, which are not abundant and not sticky, so they drag the family medians down. This is why the tight sets are the ones the verdict uses.

In [25]:
loose_tbl = val[val.kind == 'loose family (comparison only)'].groupby('set_name').agg(
    proteins=('gene_id', 'size'), median_frequency=('frequency', 'median'),
    max_frequency=('frequency', 'max'))
tight_tbl = val[val.kind == 'expected frequent flyer'].groupby('set_name').agg(
    proteins=('gene_id', 'size'), median_frequency=('frequency', 'median'),
    max_frequency=('frequency', 'max'))
print('tight (curated):'); print(tight_tbl.to_string())
print(); print('loose (keyword):'); print(loose_tbl.to_string())
tight_tbl

tight (curated):
                                        proteins  median_frequency  max_frequency
set_name                                                                         
GAPDH                                          2              0.81           0.84
HSP70/HSP90                                    3              0.88           0.94
actin                                          1              0.96           0.96
cytosolic ribosome (hyperLOPIT marker)        68              0.80           0.94
enolase                                        2              0.25           0.50
tubulin                                        6              0.61           0.96

loose (keyword):
                         proteins  median_frequency  max_frequency
set_name                                                          
actin (any)                    12              0.29           0.96
heat shock (any)               23              0.24           0.94
ribosomal protein (any)       158        

,proteins,median_frequency,max_frequency
set_name,,,
GAPDH,2,0.81,0.84
HSP70/HSP90,3,0.88,0.94
actin,1,0.96,0.96
cytosolic ribosome (hyperLOPIT marker),68,0.8,0.94
enolase,2,0.25,0.5
tubulin,6,0.61,0.96


### The test that matters for the score: does a bait come out top of its own pull-down?

Background frequency on its own is not the product; the specificity score is. The sharpest test of the score is the one case where the right answer is certain: the tagged protein itself. In every experiment with a confident bait, the bait should be near the top of its own hit list. This is run across all such experiments at once.

In [26]:
rec = []
conf_rows = []
for _, r in pull[pull.bait_confidence == 'confident'].iterrows():
    for g_, sym_ in zip(str(r.bait_gene).split(';'), str(r.bait_symbol).split(';')):
        conf_rows.append(r.copy()); conf_rows[-1]['bait_gene'] = g_; conf_rows[-1]['bait_symbol'] = sym_
for r in conf_rows:
    if r.project not in per_project:
        continue
    a_ = per_project[r.project].copy()
    tot_ = a_.intensity.sum()
    if not tot_ or not len(a_):
        continue
    a_['psm_ppm'] = a_.intensity / tot_ * 1e6
    a_ = a_[a_.n_psm >= 2]
    s = specificity(a_[['gene_id', 'n_psm', 'psm_ppm']], bg)
    s['rank'] = np.arange(1, len(s) + 1)
    hit = s[s.gene_id == r.bait_gene]
    rec.append(dict(project=r.project, bait=r.bait_symbol, bait_gene=r.bait_gene,
                    hits=len(s),
                    bait_rank=int(hit['rank'].iloc[0]) if len(hit) else np.nan,
                    bait_percentile=(100 * (1 - (hit['rank'].iloc[0] - 1) / len(s))
                                     if len(hit) else np.nan),
                    bait_specificity=float(hit.specificity.iloc[0]) if len(hit) else np.nan,
                    bait_frequency=(float(hit.background_frequency.iloc[0])
                                    if len(hit) else np.nan),
                    bait_abundance_rank=(int((a_.psm_ppm > float(hit.psm_ppm.iloc[0])).sum()) + 1
                                         if len(hit) else np.nan)))
recov = pd.DataFrame(rec)
recov = recov.merge(cl[['project', 'class']], on='project', how='left')
got = recov.dropna(subset=['bait_rank']) if len(recov) else recov
if len(got):
    print('experiments with a confident bait:', len(recov),
          '| bait detected in its own pull-down:', len(got))
    print('bait in the top 1%% of its own hit list: %d of %d'
          % (int((got.bait_percentile >= 99).sum()), len(got)))
    print('bait in the top 5%% of its own hit list: %d of %d'
          % (int((got.bait_percentile >= 95).sum()), len(got)))
    print('median bait rank: %.0f of a median %.0f proteins'
          % (got.bait_rank.median(), got.hits.median()))
    print('for comparison, ranking by raw abundance alone: median bait rank %.0f'
          % got.bait_abundance_rank.median())
    print()
    print('split by what kind of experiment it is -- the score is designed for affinity',
          'purification, where the bait is pulled out intact:')
    print(got.groupby('class').agg(experiments=('project', 'size'),
                                   in_top_5pct=('bait_percentile',
                                                lambda s: int((s >= 95).sum())),
                                   median_percentile=('bait_percentile', 'median')
                                   ).to_string())
recov

experiments with a confident bait: 30 | bait detected in its own pull-down: 29
bait in the top 1% of its own hit list: 17 of 29
bait in the top 5% of its own hit list: 21 of 29
median bait rank: 8 of a median 808 proteins
for comparison, ranking by raw abundance alone: median bait rank 31

split by what kind of experiment it is -- the score is designed for affinity purification, where the bait is pulled out intact:
                       experiments  in_top_5pct  median_percentile
class                                                             
affinity_purification           25           20          99.299065
proximity_labelling              4            1          79.537487


,project,bait,bait_gene,hits,bait_rank,bait_percentile,bait_specificity,bait_frequency,bait_abundance_rank,class
0,PXD012103,HCE1,TGME49_239010,334,1,100,9.61,0.3465,11,affinity_purification
1,PXD016383,MYR1,TGME49_254470,363,1,100,10.83,0.5248,1,affinity_purification
2,PXD016846,MORC,TGME49_305340,808,9,99.01,10.48,0.703,1,affinity_purification
3,PXD032319,ROP1,TGME49_309590,38,9,78.95,5.137,0.6634,3,affinity_purification
4,PXD032360,GRA28,TGME49_231960,428,4,99.3,7.284,0.5248,4,affinity_purification
5,PXD032362,BCC7,TGME49_311230,491,4,99.39,6.944,0.5842,2,affinity_purification
6,PXD037318,TgH1-like,TGME49_315570,65,4,95.38,7.077,0.2475,14,affinity_purification
7,PXD038848,TgBDP1,TGME49_263580,244,2,99.59,10.45,0.4059,1,affinity_purification
8,PXD039432,CDPK1,TGME49_301440,561,NaN,NaN,NaN,NaN,NaN,affinity_purification
9,PXD039896,C3,TGME49_286470,1167,11,99.14,7.803,0.2673,48,affinity_purification


### Where the known complexes do best

No experiment here baits the glideosome, AMA1-RON2, MIC2-M2AP or MORC/HDAC3, so there is no pull-down in which they *should* be specific. What can be asked is the next best thing: across the experiments we have, is there any in which the score marks them as specific, and do the members of one complex peak together?

In [27]:
comp_genes = val[val.kind == 'known complex'][['set_name', 'symbol', 'gene_id']]
best = []
for p in bg_projects:
    a_ = per_project[p].copy()
    tot_ = a_.intensity.sum()
    if not tot_ or not len(a_):
        continue
    a_['psm_ppm'] = a_.intensity / tot_ * 1e6
    a_ = a_[a_.n_psm >= 2]
    s = specificity(a_[['gene_id', 'n_psm', 'psm_ppm']], bg)
    s['rank'] = np.arange(1, len(s) + 1)
    s['percentile'] = 100 * (1 - (s['rank'] - 1) / len(s))
    m = comp_genes.merge(s[['gene_id', 'rank', 'percentile', 'specificity']], on='gene_id')
    m['project'] = p
    best.append(m)
cb = pd.concat(best, ignore_index=True)
top = cb.sort_values('specificity', ascending=False).groupby(
    ['set_name', 'symbol'], as_index=False).first()
print('best specificity each complex member reaches in any of the', N_EXP, 'experiments,')
print('and the project in which it reaches it:')
top.sort_values(['set_name', 'specificity'], ascending=[True, False])[
    ['set_name', 'symbol', 'project', 'rank', 'percentile', 'specificity']]

best specificity each complex member reaches in any of the 50 experiments,
and the project in which it reaches it:


,set_name,symbol,project,rank,percentile,specificity
0,AMA1-RON,AMA1,PXD011692,26,58.33,4.643
1,AMA1-RON,RON2,PXD043994,127,88.39,3.813
2,AMA1-RON,RON4,PXD043994,262,75.94,2.775
4,MIC2-M2AP,MIC2,PXD032360,72,83.41,4.919
3,MIC2-M2AP,M2AP,PXD043924,212,88.11,3.451
6,MORC/HDAC3,MORC,PXD016846,9,99.01,10.48
5,MORC/HDAC3,HDAC3,PXD016846,47,94.31,5.284
10,MyoA glideosome,GAP50,PXD012103,9,97.6,5.746
9,MyoA glideosome,GAP45,PXD012103,11,97.01,5.583
8,MyoA glideosome,GAP40,PXD039919,44,94.26,4.946


In [28]:
# do the members of one complex peak in the same experiment?
pk = cb.loc[cb.groupby(['set_name', 'symbol']).specificity.idxmax()]
agree = pk.groupby('set_name').agg(members=('symbol', 'size'),
                                   distinct_best_projects=('project', 'nunique'),
                                   top_project=('project', lambda s: s.mode().iloc[0]),
                                   members_sharing_it=('project',
                                                       lambda s: int((s == s.mode().iloc[0]).sum())))
agree

,members,distinct_best_projects,top_project,members_sharing_it
set_name,,,,
AMA1-RON,3,2,PXD043994,2
MIC2-M2AP,2,2,PXD032360,1
MORC/HDAC3,2,1,PXD016846,2
MyoA glideosome,6,3,PXD012103,4


### Why the number of independent baits is the limit

The frequency of a protein can only take one of `N+1` values when there are `N` experiments, and a protein that happens to be in two of the few baited complexes will look like a frequent flyer. The question below is whether the experiments we have are really independent looks at the background, or several looks at the same biology.

In [29]:
print('experiments in the background:', N_EXP)
print('of which a confident bait gene was identified:',
      int((pull.bait_confidence == 'confident').sum()))
print('distinct confident bait genes:',
      pull.loc[pull.bait_confidence == 'confident', 'bait_gene'].nunique())
print('smallest frequency step the table can resolve: 1/%d = %.3f' % (N_EXP, 1 / N_EXP))
print()
bait_comp = pull.merge(nodes.rename(columns={'compartment': 'bait_compartment'})[
    ['gene_id', 'bait_compartment']], left_on='bait_gene', right_on='gene_id', how='left')
print('compartments the baits come from:')
print(bait_comp.bait_compartment.fillna('(unknown)').value_counts().to_string())
print()
print('proteins detected in only one experiment: %d of %d (%.0f%%)'
      % (int((bg.experiments_detected == 1).sum()), len(bg),
         100 * (bg.experiments_detected == 1).mean()))
bg.frequency.describe()

experiments in the background: 50
of which a confident bait gene was identified: 29
distinct confident bait genes: 25
smallest frequency step the table can resolve: 1/50 = 0.020

compartments the baits come from:
bait_compartment
(unknown)                    20
unassigned                   13
nucleus - chromatin           7
dense granules                4
rhoptries 1                   1
IMC                           1
micronemes                    1
cytosol                       1
ER 2                          1
mitochondrion - membranes     1

proteins detected in only one experiment: 832 of 4952 (17%)


count    4952.000000
mean        0.187080
std         0.197356
min         0.020000
25%         0.040000
50%         0.100000
75%         0.260000
max         0.960000

## 9 · Files written

In [30]:
cl_out = cl[['project', 'class', 'confidence', 'decided_by', 'cue', 'rule_class',
             'bait_symbol', 'bait_gene', 'bait_product', 'bait_confidence', 'bait_how',
             'n_runs',
             'acquisition', 'psm_rows', 'psm_1pct', 'genes_2psm', 'has_lfq',
             'title', 'note', 'example_runs']].sort_values(['class', 'project'])
cl_out.to_csv(os.path.join(OUT, 'project_classification.tsv'), sep='\t', index=False)

bg_out = bg[['gene_id', 'product_name', 'experiments_detected', 'n_experiments',
             'frequency', 'median_abundance', 'max_abundance', 'n_experiments_with_lfq',
             'median_psm_ppm', 'max_psm_ppm', 'total_psm', 'runs_detected',
             'compartment']].sort_values('frequency', ascending=False)
strict = variants['strict (>=3 matches, >=2 peptides)'].set_index('gene_id')
loose = variants['loose (>=1 match)'].set_index('gene_id')
bg_out['frequency_strict'] = bg_out.gene_id.map(strict.frequency).fillna(0.0)
bg_out['frequency_loose'] = bg_out.gene_id.map(loose.frequency).fillna(0.0)
bg_out.to_csv(os.path.join(OUT, 'background.tsv'), sep='\t', index=False)

scored.to_csv(os.path.join(OUT, 'specificity_example.tsv'), sep='\t', index=False)
val.sort_values(['kind', 'set_name', 'frequency'], ascending=[True, True, False]).to_csv(
    os.path.join(OUT, 'validation.tsv'), sep='\t', index=False)
recov.to_csv(os.path.join(OUT, 'bait_recovery.tsv'), sep='\t', index=False)
top.to_csv(os.path.join(OUT, 'complex_best_specificity.tsv'), sep='\t', index=False)

coverage = pd.DataFrame([
    dict(item='projects selected for the uniform re-search', n=len(selected)),
    dict(item='project directories on disk', n=len(disk)),
    dict(item='finished (done.json receipt)', n=int(disk.finished.sum())),
    dict(item='excluded: unsupported.json', n=int(disk.unsupported.sum())),
    dict(item='excluded: NOT_COMPARABLE.json (HDX-MS)', n=int(disk.not_comparable.sum())),
    dict(item='finished and classified', n=len(cl)),
    dict(item='still waiting in the queue', n=len(selected) - int(disk.finished.sum())),
    dict(item='experiments in the background (affinity + proximity)', n=N_EXP),
    dict(item='of those with a confident bait gene',
         n=int((pull.bait_confidence == 'confident').sum())),
    dict(item='distinct confident bait genes',
         n=int(pull.loc[pull.bait_confidence == 'confident', 'bait_gene'].nunique())),
    dict(item='runs behind those experiments', n=n_runs_total),
    dict(item='PSM rows read', n=int(ps.psm_rows.sum())),
    dict(item='PSMs at 1% FDR, target', n=int(ps.psm_1pct.sum())),
    dict(item='parasite proteins in the background table', n=len(bg)),
] + [dict(item=f'class: {k}', n=int(v)) for k, v in cl.groupby('class').size().items()])
coverage.to_csv(os.path.join(OUT, 'coverage.tsv'), sep='\t', index=False)
for f in sorted(os.listdir(OUT)):
    print(f, os.path.getsize(os.path.join(OUT, f)), 'bytes')
coverage

background.tsv 788855 bytes
bait_recovery.tsv 3643 bytes
complex_best_specificity.tsv 1111 bytes
coverage.tsv 662 bytes
project_classification.tsv 39417 bytes
specificity_example.tsv 79990 bytes
validation.tsv 50786 bytes


,item,n
0,projects selected for the uniform re-search,203
1,project directories on disk,111
2,finished (done.json receipt),97
3,excluded: unsupported.json,12
4,excluded: NOT_COMPARABLE.json (HDX-MS),2
5,finished and classified,94
6,still waiting in the queue,106
7,experiments in the background (affinity + proximity),50
8,of those with a confident bait gene,29
9,distinct confident bait genes,25


## 10 · Verdict

The verdict cell prints the three numbers the decision rests on: how far apart the known complexes and the frequent flyers are, how many independent baits stand behind the table, and what happens to each known complex at a usable cutoff.

In [31]:
lines = []
lines.append('Experiments: %d (%d affinity purification, %d proximity labelling) out of %d '
             'selected PRIDE projects; %d finished so far.'
             % (N_EXP, int((cl["class"] == 'affinity_purification').sum()),
                int((cl['class'] == 'proximity_labelling').sum()), len(selected),
                int(disk.finished.sum())))
lines.append('Background table: %d parasite proteins.' % len(bg))
lines.append('Known complex members: median background frequency %.2f (n=%d).'
             % (a.median(), len(a)))
lines.append('Expected frequent flyers: median background frequency %.2f (n=%d).'
             % (b.median(), len(b)))
lines.append('Separation: AUC %.3f, p = %.2g.' % (auc, u.pvalue))
lines.append('At a 50%% frequency cutoff, %d of %d complex members would be discarded as '
             'background and %d of %d flyers correctly caught.'
             % (int((a >= 0.5).sum()), len(a), int((b >= 0.5).sum()), len(b)))
lines.append('No cutoff removes half the frequent flyers while keeping 85% of the known '
             'complex members.' if not FREQ_FILTER_WORKS else
             'A frequency cutoff exists that removes half the frequent flyers while '
             'keeping 85% of the known complex members.')
lines.append('Independent confident baits: %d (over %d experiments).'
             % (pull.loc[pull.bait_confidence == 'confident', 'bait_gene'].nunique(),
                int((pull.bait_confidence == 'confident').sum())))
ap_got = got[got['class'] == 'affinity_purification'] if len(got) else got
SCORE_WORKS = bool(len(ap_got) >= 5) and float((ap_got.bait_percentile >= 95).mean()) >= 0.6
if len(got):
    lines.append('Bait recovery by the full score: the tagged protein is in the top 5%% of '
                 'its own hit list in %d of %d experiments with a confident bait, and in '
                 '%d of %d of the affinity-purification ones.'
                 % (int((got.bait_percentile >= 95).sum()), len(got),
                    int((ap_got.bait_percentile >= 95).sum()), len(ap_got)))
lines.append('VERDICT: the frequency column alone is %s as a filter. '
             'The full specificity score %s.'
             % ('NOT usable' if not FREQ_FILTER_WORKS else 'usable',
                'recovers the bait in its own pull-down and is usable for ranking'
                if SCORE_WORKS else 'is not yet demonstrated'))
print('\n'.join('- ' + x for x in lines))

- Experiments: 50 (40 affinity purification, 10 proximity labelling) out of 203 selected PRIDE projects; 97 finished so far.
- Background table: 4952 parasite proteins.
- Known complex members: median background frequency 0.70 (n=13).
- Expected frequent flyers: median background frequency 0.80 (n=82).
- Separation: AUC 0.689, p = 0.014.
- At a 50% frequency cutoff, 12 of 13 complex members would be discarded as background and 74 of 82 flyers correctly caught.
- No cutoff removes half the frequent flyers while keeping 85% of the known complex members.
- Independent confident baits: 25 (over 29 experiments).
- Bait recovery by the full score: the tagged protein is in the top 5% of its own hit list in 21 of 29 experiments with a confident bait, and in 20 of 25 of the affinity-purification ones.
- VERDICT: the frequency column alone is NOT usable as a filter. The full specificity score recovers the bait in its own pull-down and is usable for ranking.
